# Búsqueda local

Con la solución del greedy, buscamos ahora hacer una búsqueda local para intentar mejorar los resultados

In [1]:
import pandas as pd 
from importlib import reload

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")

cajas_nuevas = pd.read_csv("4r.cajas_nuevas.csv")
factibilidad = pd.read_csv("Factibilidad/factibilidad_3mm.csv")

solucion = pd.read_csv('Soluciones/solucion7-greedy_3mm_mejor_sol_mas_cajas.csv')

Empecemos guardando los productos y tipos de cajas en listas en el estado actual, para cargarlos luego a las soluciones. Almacenamos también las cajas asignables a cada producto en un diccionario.

In [2]:
def guardar_cajas_y_productos(grosor=3):
    
    cajas = {
        row["caja_tipo_id"]: Caja(
            caja_id=row["caja_tipo_id"],
            dim_interior_ancho=row["caja_interior_ancho"],
            dim_interior_largo=row["caja_interior_largo"],
            dim_interior_alto=row["caja_interior_alto"]
        )
        for _, row in cajas_nuevas.iterrows()
    }

    prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")
    productos = {
        row["codigo_producto"]: Producto(
            codigo_producto = row['codigo_producto'],
            cantidad_paquetes = row['cantidad_paquetes'],
            peso_paquete = row['peso_neto_paquete'],
            demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
            demanda_curitiba = row['volumen_producto_planta_curitiba'],
            demanda_santiago = row['volumen_producto_planta_santiago'],
            demanda_monterrey = row['volumen_producto_planta_monterrey'],
            demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
            dim_producto_ancho = row['dim_producto_ancho'], 
            dim_producto_largo = row['dim_producto_largo'],
            dim_producto_alto = row['dim_producto_alto']
        )
        for _, row in prod_op_merge.iterrows()
    }
    
    cajas_asignables_por_producto = {}

    for codigo, group in factibilidad.groupby('codigo_producto'):
        # Obtener IDs de los tipos de cajas
        cajas_ids_unicos = list(group['caja_tipo_id'].unique())
        
        cajas_producto = []
        for caja_id in cajas_ids_unicos:
            cajas_producto.append(caja_id)
            
        cajas_asignables_por_producto[codigo] = cajas_producto
                
    # Elegir grosor
    for caja_id, caja in cajas.items():
        caja.elegir_grosor(grosor_mm=grosor)
        
    return cajas, productos, cajas_asignables_por_producto

#### **Reconstrucción de la solución inicial (Greedy)**

El csv exportado por `exportar_submmit` solo guarda las dimensiones *exteriores* de la caja asignada a cada producto, no el `caja_tipo_id`. Para poder operar con objetos `Caja` (y sus descuentos por volumen), reconstruimos el `caja_tipo_id` real restando el grosor a las dimensiones exteriores y buscando la caja correspondiente entre las cajas ya creadas.

In [3]:
grosor = 3
cajas, productos, cajas_asignables_por_producto = guardar_cajas_y_productos(grosor=grosor)

# Índice de cajas por sus dimensiones interiores (redondeadas), para poder
# encontrar el caja_tipo_id real a partir de las dimensiones exteriores del csv
indice_cajas_por_dim = {
    (c.dim_interior_ancho, c.dim_interior_largo, c.dim_interior_alto): c
    for c in cajas.values()
}

solucion_inicial = Solucion(grosor, "Greedy 5 (maximizar utilización de pallet) - punto de partida")

for _, row in solucion.iterrows():
    producto = productos[row["codigo_producto"]]

    dim_ancho = row["caja_exterior_ancho"] - 2 * row["caja_grosor_mm"]
    dim_largo = row["caja_exterior_largo"] - 2 * row["caja_grosor_mm"]
    dim_alto = row["caja_exterior_alto"] - 2 * row["caja_grosor_mm"]

    caja = indice_cajas_por_dim[(dim_ancho, dim_largo, dim_alto)]

    asignacion = Asignacion(producto, caja)
    solucion_inicial.agregar_asignacion(asignacion)

solucion_inicial.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Greedy 5 (maximizar utilización de pallet) - punto de partida
Número de tipos de cajas distintos: 76
Costo packaging: 27406334.33999998
Costo flete: 161380650
Costo total: 188786984.33999997
Utilización de pallet promedio: 0.9593515126880586
Utilización de caja promedio: 0.9445527162171283
Ahorro costo total: 9.77279%


#### **Cálculo del delta de costo de un movimiento**

El costo de packaging de una caja depende del volumen acumulado de **todos** los productos que la usan (por los descuentos por volumen). Por eso, mover un producto de una caja a otra no tiene un delta de costo aislado: afecta también el costo de los demás productos que ya comparten esa caja.

La función `calcular_delta_costo` simula el movimiento (usando los métodos ya existentes `asignar_producto` / `revocar_producto` de `Caja`, que actualizan los descuentos), mide el cambio real en el costo total de las dos cajas involucradas, y **revierte la simulación** antes de devolver el resultado. El costo de flete sí es independiente del resto de los productos de la caja (depende solo de la demanda del producto y de `cantidad_cajas_por_pallet` de la caja), así que se calcula aparte, sin necesidad de simular nada.

Nota: no volvemos a chequear factibilidad (dimensión, headspace, resistencia, utilización de pallet ≥ 0.6) porque `cajas_asignables_por_producto` ya viene filtrado por esos criterios desde `5_factibilidad.ipynb`.

In [7]:
def calcular_delta_costo(producto, caja_actual, caja_nueva):
    """
    Calcula cuánto cambiaría el costo total (packaging + flete) si se
    reasignara `producto` de `caja_actual` a `caja_nueva`.
    Simula el movimiento y lo revierte: no deja efectos secundarios.
    """
    # --- Costo de packaging (afecta a todos los productos de ambas cajas) ---
    costo_antes = caja_actual.costo_packaging_total() + caja_nueva.costo_packaging_total()

    caja_actual.revocar_producto(producto)
    caja_nueva.asignar_producto(producto)

    costo_despues = caja_actual.costo_packaging_total() + caja_nueva.costo_packaging_total()

    # Revertimos la simulación para dejar el estado como estaba
    caja_nueva.revocar_producto(producto)
    caja_actual.asignar_producto(producto)

    delta_packaging = costo_despues - costo_antes

    # --- Costo de flete (independiente de otros productos de la caja) ---
    pallets_actual = Asignacion(producto, caja_actual).cant_pallets_requeridas()
    pallets_nueva = Asignacion(producto, caja_nueva).cant_pallets_requeridas()
    delta_flete = 150 * (pallets_nueva - pallets_actual)

    return delta_packaging + delta_flete

#### **Búsqueda local: mejor mejora por pasada**

En cada pasada se evalúa reasignar **cada producto** a cada una de sus cajas alternativas factibles, y se aplica el movimiento con la mayor reducción de costo total encontrado en esa pasada. Se repite hasta que no se encuentra ninguna mejora (óptimo local) o se alcanza `max_iteraciones`.

Por ahora el único tipo de movimiento es "reasignar un producto a otra caja"; más adelante se puede extender con otros vecindarios (por ejemplo, intercambio de cajas entre pares de productos).

⚠️ **Nota de performance:** según `5_factibilidad.ipynb`, algunos productos tienen miles de cajas candidatas asignables. Evaluar *todas* las combinaciones producto×caja en cada pasada puede ser lento en instancias grandes. Si se vuelve impráctico, se puede limitar la cantidad de candidatos evaluados por producto por pasada (parámetro `max_candidatos_por_producto`, que si se deja en `None` evalúa todos).

In [ ]:
import random

def busqueda_local(solucion, cajas, cajas_asignables_por_producto, max_iteraciones=50,
                    max_candidatos_por_producto=None, verbose=True, semilla=42):
    """
    Búsqueda local por 'mejor mejora' (steepest descent) sobre reasignaciones
    individuales de caja. Modifica `solucion` in-place y devuelve un DataFrame
    con el historial de movimientos aplicados.

    Si se interrumpe manualmente (KeyboardInterrupt), los movimientos ya
    aplicados quedan y se devuelve igual el historial parcial armado hasta
    ese momento, en vez de perderse.
    """
    random.seed(semilla)
    historial = []
    costo_actual = solucion.costo_total()
    UMBRAL_MEJORA = 1e-6  # tolerancia para evitar ciclos por ruido de punto flotante
    iteracion = 0

    try:
        for iteracion in range(1, max_iteraciones + 1):
            mejor_delta = -UMBRAL_MEJORA
            mejor_movimiento = None  # (asignacion, caja_nueva)

            for asignacion in solucion.asignaciones:
                producto = asignacion.producto
                caja_actual = asignacion.caja
                candidatos = cajas_asignables_por_producto.get(producto.codigo_producto, [])

                if max_candidatos_por_producto is not None and len(candidatos) > max_candidatos_por_producto:
                    candidatos = random.sample(candidatos, max_candidatos_por_producto)

                for caja_id_candidata in candidatos:
                    if caja_id_candidata == caja_actual.caja_id:
                        continue

                    caja_candidata = cajas[caja_id_candidata]
                    delta = calcular_delta_costo(producto, caja_actual, caja_candidata)

                    if delta < mejor_delta:
                        mejor_delta = delta
                        mejor_movimiento = (asignacion, caja_candidata)

            if mejor_movimiento is None:
                if verbose:
                    print(f"Iteración {iteracion}: no se encontraron mejoras. Óptimo local alcanzado.")
                break

            # Aplicamos el mejor movimiento encontrado en esta pasada
            asignacion, caja_nueva = mejor_movimiento
            producto = asignacion.producto
            caja_vieja = asignacion.caja

            caja_vieja.revocar_producto(producto)
            caja_nueva.asignar_producto(producto)
            asignacion.caja = caja_nueva

            # Actualizamos el tracking de tipos de caja utilizados en la solución
            if caja_nueva not in solucion.tipos_cajas_utilizados:
                solucion.tipos_cajas_utilizados.append(caja_nueva)
                solucion.cantidad_tipos_cajas += 1
            if len(caja_vieja.productos_asignados) == 0 and caja_vieja in solucion.tipos_cajas_utilizados:
                solucion.tipos_cajas_utilizados.remove(caja_vieja)
                solucion.cantidad_tipos_cajas -= 1

            costo_actual += mejor_delta

            historial.append({
                "iteracion": iteracion,
                "codigo_producto": producto.codigo_producto,
                "caja_anterior": caja_vieja.caja_id,
                "caja_nueva": caja_nueva.caja_id,
                "delta_costo": mejor_delta,
                "costo_total_estimado": costo_actual
            })

            if verbose:
                print(f"Iteración {iteracion}: {producto.codigo_producto} "
                      f"{caja_vieja.caja_id} -> {caja_nueva.caja_id} "
                      f"(Δ={mejor_delta:,.2f} | costo total ≈ {costo_actual:,.2f})")

    except KeyboardInterrupt:
        if verbose:
            print(f"\nInterrumpido manualmente en la iteración {iteracion}. "
                  f"Se conservan los {len(historial)} movimientos ya aplicados.")

    return pd.DataFrame(historial)

#### **Ejecutamos la búsqueda local**

In [8]:
historial_busqueda_local = busqueda_local(
    solucion_inicial,
    cajas,
    cajas_asignables_por_producto,
    max_iteraciones=50,          # subir si todavía encuentra mejoras al llegar al límite
    max_candidatos_por_producto=None,  # poner un número (ej. 200) si es muy lento
    verbose=True
)

historial_busqueda_local

Iteración 1: BR0239 CAJ2229012 -> CAJ2132580 (Δ=-50,991.00 | costo total ≈ 188,735,993.34)
Iteración 2: BR0321 CAJ1942558 -> CAJ1955788 (Δ=-49,950.00 | costo total ≈ 188,686,043.34)
Iteración 3: BR0159 CAJ0708749 -> CAJ0443990 (Δ=-36,403.14 | costo total ≈ 188,649,640.20)
Iteración 4: BR0079 CAJ1943342 -> CAJ1927270 (Δ=-29,858.10 | costo total ≈ 188,619,782.10)
Iteración 5: BR0347 CAJ1945596 -> CAJ1927270 (Δ=-29,401.62 | costo total ≈ 188,590,380.48)
Iteración 6: BR0147 CAJ1431292 -> CAJ1399148 (Δ=-22,808.58 | costo total ≈ 188,567,571.90)
Iteración 7: BR0160 CAJ0708749 -> CAJ0443892 (Δ=-17,677.98 | costo total ≈ 188,549,893.92)
Iteración 8: BR0240 CAJ2229012 -> CAJ2132580 (Δ=-12,971.70 | costo total ≈ 188,536,922.22)
Iteración 9: BR0361 CAJ0708749 -> CAJ0443892 (Δ=-12,189.48 | costo total ≈ 188,524,732.74)


KeyboardInterrupt: 

#### **Comparación: Greedy vs. Búsqueda local**

In [9]:
solucion_inicial.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Greedy 5 (maximizar utilización de pallet) - punto de partida
Número de tipos de cajas distintos: 78
Costo packaging: 27296782.739999976
Costo flete: 161227950
Costo total: 188524732.73999998
Utilización de pallet promedio: 0.9579706456578424
Utilización de caja promedio: 0.9471952523608305
Ahorro costo total: 9.89813%


Exportamos la solución mejorada al mismo formato usado por las soluciones greedy:

In [10]:
solucion_inicial.exportar_submmit(nombre_csv="7-busqueda_local_3mm")